# Analiza zahtevka za stroške

Ta zvezek prikazuje, kako ustvariti agente, ki uporabljajo vtičnike za obdelavo potnih stroškov iz lokalnih slik računov, generiranje e-pošte za zahtevek za stroške in vizualizacijo podatkov o stroških z uporabo tortnega diagrama. Agenti dinamično izbirajo funkcije glede na kontekst naloge.

Koraki:
1. OCR agent obdela lokalno sliko računa in izlušči podatke o potnih stroških.
2. E-poštni agent generira e-pošto za zahtevek za stroške.

### Primer scenarija potnih stroškov:
Predstavljajte si, da ste zaposleni, ki potuje na poslovno srečanje v drugo mesto. Vaše podjetje ima politiko povračila vseh razumnih stroškov, povezanih s potovanjem. Tukaj je razčlenitev morebitnih potnih stroškov:
- Prevoz:
Letalske vozovnice za povratno pot od domačega mesta do cilnega mesta.
Taksi ali storitve prevoza na zahtevo do in iz letališča.
Lokalni prevoz v ciljnem mestu (kot so javni prevoz, najem avtomobila ali taksiji).

- Nastanitev:
Bivanje v hotelu tri noči v srednje kakovostnem poslovnem hotelu blizu kraja srečanja.

- Obroki:
Dnevni dodatek za obroke za zajtrk, kosilo in večerjo, glede na politiko dnevnic podjetja.

- Razni stroški:
Parkirnine na letališču.
Stroški interneta v hotelu.
Napitnine ali majhni stroški za storitve.

- Dokumentacija:
Predložite vse račune (lete, taksije, hotel, obroke itd.) in izpolnjen obrazec za povračilo stroškov.


## Uvozi potrebne knjižnice

Uvozi potrebne knjižnice in module za beležko.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Določi modele stroškov

 Ustvari Pydantic model za posamezne stroške in razred ExpenseFormatter za pretvorbo uporabnikove poizvedbe v strukturirane podatke o stroških.

 Vsak strošek bo predstavljen v formatu:
 `{'date': '07-Mar-2025', 'description': 'let do cilja', 'amount': 675.99, 'category': 'Prevoz'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Določanje orodij - ustvarjanje elektronske pošte

Ustvarite funkcijo orodja za generiranje e-pošte za oddajo zahtevka za povračilo stroškov.  
- To orodje uporablja dekorator `@tool` iz Microsoft Agent Framework.  
- Izračuna skupni znesek stroškov in podrobnosti oblikuje v vsebino e-pošte.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Orodje za izvleček potnih stroškov iz slik računov

Ustvarite funkcijo orodja za izvleček potnih stroškov iz slik računov.
- To orodje uporablja dekorator `@tool` iz Microsoft Agent Framework.
- Prebere sliko računa, jo kodira kot base64 in vrne podatkovni URI za analizo agentu.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Obdelava stroškov

Določite agente in jih povežite v zaporedno delovno tok z uporabo `WorkflowBuilder`.
- OCR agent iz slike računa s pomočjo orodja `load_receipt_image` izlušči strukturirane podatke o stroških.
- Email agent vzame izluščene podatke in ustvari profesionalno e-pošto za uveljavljanje stroškov z uporabo orodja `generate_expense_email`.
- `WorkflowBuilder` z `add_edge` ustvari zaporedno verigo: OCR agent → Email agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Glavna funkcija

Zgradite zaporedni potek dela in ga zaženite za obdelavo slike računa ter ustvarjanje e-pošte za uveljavitev stroškov.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo storitve za prevajanje z umetno inteligenco [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, prosimo, upoštevajte, da lahko avtomatizirani prevodi vsebujejo napake ali netočnosti. Izvirni dokument v svojem izvirnem jeziku se smatra za avtoritativni vir. Za kritične informacije priporočamo strokovni človeški prevod. Za morebitna nesporazumevanja ali napačne razlage, ki izhajajo iz uporabe tega prevoda, ne prevzemamo odgovornosti.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
